# Lesson 8.4: Multiple Regression: Many Predictors at Once

Predict house price from square feet and bedrooms together, and see why the bedrooms effect disappears once size is held fixed, using the lesson's ten houses.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Enter the ten houses

Square feet, bedrooms and price (in thousands of dollars). Bigger houses tend to have more bedrooms, so the two predictors are tangled together. That is the whole point of this lab.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.DataFrame({
    'sqft': [1000, 1200, 1400, 1600, 1800, 2000, 2200, 2400, 2600, 2800],
    'bedrooms': [2, 2, 3, 3, 4, 4, 4, 5, 5, 6],
    'price': [250, 280, 310, 350, 390, 420, 450, 480, 520, 550]})

print(df.corr().round(3))

## 2. Bedrooms alone

On its own, bedrooms seems to matter a lot: each extra bedroom goes with a large jump in price. But bedrooms may just be standing in for size.

In [2]:
alone = smf.ols('price ~ bedrooms', data=df).fit()

print(f'bedrooms alone: {alone.params["bedrooms"]:.1f} thousand per bedroom')
print(f'R squared = {alone.rsquared:.3f}')

## 3. Both predictors together

Adding sqft to the formula with + fits one model with two predictors. Each coefficient is the change in price for one more unit of that predictor while the other is held fixed.

In [3]:
both = smf.ols('price ~ sqft + bedrooms', data=df).fit()

print(both.params.round(4))
print()
print(both.pvalues.round(4))

## 4. Read the fit and what 'held fixed' means

The sqft coefficient compares two houses with the same number of bedrooms and different sizes. The bedrooms coefficient compares houses of the same size with different bedrooms. R squared says how much of the price variation the pair explains.

In [4]:
print(f'R squared          = {both.rsquared:.4f}')
print(f'adjusted R squared = {both.rsquared_adj:.4f}')
print(f'overall F = {both.fvalue:.1f}, p = {both.f_pvalue:.2e}')

new = pd.DataFrame({'sqft': [1500], 'bedrooms': [3]})
print(f'predicted price, 1500 sqft, 3 bed = {both.predict(new)[0]:.1f}')

## Try this

- Predict the price of a 1,500 sq ft house with 5 bedrooms, then compare with 3 bedrooms. How different are they, and why so small?
- Drop sqft from the formula and refit with only bedrooms. How does the bedrooms coefficient change?

## The same idea in R, SPSS and Stata

**R**

```r
sqft <- c(1000, 1200, 1400, 1600, 1800, 2000, 2200, 2400, 2600, 2800)
bedrooms <- c(2, 2, 3, 3, 4, 4, 4, 5, 5, 6)
price <- c(250, 280, 310, 350, 390, 420, 450, 480, 520, 550)
summary(lm(price ~ bedrooms))
fit <- lm(price ~ sqft + bedrooms)
summary(fit)
predict(fit, data.frame(sqft = 1500, bedrooms = 3))
```

**SPSS** (Analyze > Regression > Linear (Dependent: price, Independents: sqft and bedrooms))

```
DATA LIST FREE / sqft bedrooms price.
BEGIN DATA
1000 2 250  1200 2 280  1400 3 310  1600 3 350  1800 4 390
2000 4 420  2200 4 450  2400 5 480  2600 5 520  2800 6 550
END DATA.
REGRESSION
  /STATISTICS COEFF R ANOVA
  /DEPENDENT price
  /METHOD=ENTER sqft bedrooms.
```

**Stata**

```
clear
set obs 10
generate sqft = 800 + 200 * _n
generate bedrooms = real(word("2 2 3 3 4 4 4 5 5 6", _n))
generate price = real(word("250 280 310 350 390 420 450 480 520 550", _n))
regress price bedrooms
regress price sqft bedrooms
display _b[_cons] + _b[sqft] * 1500 + _b[bedrooms] * 3
```